# 第 9 课：后台任务 — 异步队列与任务链

在前面的课程中，我们了解了 AI 记忆系统的各个层次和数据结构。但有一个现实问题：**很多操作非常耗时**。

比如用户上传一张截图，系统需要：OCR 识别文字 → AI 提取关键信息 → 写入数据库 → 更新记忆层……整个流程可能要 10-30 秒。

我们不可能让用户干等着。这就引出了本课的主题：**后台任务**。

---

## 9.1 为什么需要后台任务

### 一次截图识别的完整流程

```
用户上传截图
    ↓ (~2s)
OCR 文字识别
    ↓ (~3s)
AI 提取关键信息
    ↓ (~5s)
写入数据库
    ↓ (~1s)
跑记忆层更新
    ↓ (~10s)
完成！

总计：~21 秒
```

如果这些全部在用户请求的主线程里做，用户就要盯着 loading 转圈 20 多秒。这体验太差了。

### 前端类比：Web Worker

如果你写过前端，一定知道 **Web Worker**：
- 主线程负责 UI 交互，不能被阻塞
- 耗时计算扔给 Web Worker 在后台跑
- 跑完了通过 `postMessage` 通知主线程

后台任务的思路完全一样：
- **API 服务**（主线程）：接收用户请求，立刻返回「收到了」
- **后台 Worker**（Web Worker）：慢慢处理耗时操作
- 处理完了更新状态，前端来查询结果

In [ ]:
import time

def process_screenshot_sync(image_path: str) -> dict:
    """同步处理截图 — 用户必须等待全部完成"""
    print(f"开始处理: {image_path}")
    
    print("  [1/5] OCR 文字识别...")
    time.sleep(2)  # 模拟耗时
    
    print("  [2/5] AI 提取关键信息...")
    time.sleep(3)
    
    print("  [3/5] 写入数据库...")
    time.sleep(1)
    
    print("  [4/5] 更新事实记忆层...")
    time.sleep(2)
    
    print("  [5/5] 更新记忆层...")
    time.sleep(3)
    
    return {"status": "completed", "facts_found": 5}

# 同步方式：用户必须等待 11 秒
start = time.time()
result = process_screenshot_sync("meeting_notes.png")
elapsed = time.time() - start
print(f"\n完成！耗时 {elapsed:.1f} 秒")
print(f"结果: {result}")
print(f"\n问题：用户等了 {elapsed:.1f} 秒才得到响应，体验很差！")

---
## 9.2 消息队列的概念

### 生产者-消费者模型

后台任务的核心是 **消息队列**（Message Queue），它用的是「生产者-消费者」模型。

### 餐厅类比

想象你去餐厅吃饭：

| 餐厅 | 消息队列系统 |
|------|-------------|
| 你（顾客）| 生产者 Producer — 提交任务的人 |
| 点菜单 | 消息/任务 Message — 要做什么的描述 |
| 传菜窗口的订单队列 | 队列 Queue — 存放待处理任务 |
| 厨师 | 消费者 Consumer/Worker — 实际干活的人 |
| 取餐号 | 任务 ID — 用来查询进度 |

关键点：**你下了单拿到取餐号就可以去座位等着了，不需要站在厨房门口盯着厨师做菜**。

### 常见的消息队列

| 队列 | 特点 | 适合场景 |
|------|------|----------|
| **Redis** | 简单快速，内存级 | 中小项目，任务量不大 |
| **RabbitMQ** | 功能丰富，可靠性高 | 需要复杂路由的场景 |
| **Kafka** | 超高吞吐，持久化 | 大数据、日志处理 |

我们先用 Python 自带的 `queue` 模块来理解基本概念。

In [ ]:
import queue
import threading
import time
import uuid

# 创建一个任务队列
task_queue: queue.Queue = queue.Queue()

# 存储任务结果
task_results: dict[str, dict] = {}


def producer_submit_task(task_data: str) -> str:
    """生产者：提交任务到队列，立刻返回任务 ID"""
    task_id = str(uuid.uuid4())[:8]
    task_results[task_id] = {"status": "pending", "data": None}
    task_queue.put({"id": task_id, "data": task_data})
    print(f"[生产者] 提交任务 {task_id}，排队等候处理")
    return task_id  # 立刻返回，不等待处理完成


def consumer_worker():
    """消费者：从队列中取任务并处理"""
    while True:
        try:
            task = task_queue.get(timeout=5)  # 等待新任务，5秒超时
        except queue.Empty:
            print("[消费者] 没有更多任务了，下班！")
            break
        
        task_id = task["id"]
        print(f"[消费者] 开始处理任务 {task_id}: {task['data']}")
        task_results[task_id]["status"] = "in_progress"
        
        time.sleep(2)  # 模拟耗时处理
        
        task_results[task_id] = {
            "status": "completed",
            "data": f"处理完成: {task['data']} 的结果"
        }
        print(f"[消费者] 任务 {task_id} 完成!")
        task_queue.task_done()


# 启动消费者线程（后台 Worker）
worker_thread = threading.Thread(target=consumer_worker, daemon=True)
worker_thread.start()

# 生产者快速提交 3 个任务 —— 每次提交都是瞬间完成的
print("=" * 50)
print("模拟：用户连续上传 3 张截图")
print("=" * 50)

task_ids = []
for i in range(3):
    tid = producer_submit_task(f"screenshot_{i+1}.png")
    task_ids.append(tid)

print(f"\n[前端] 所有任务已提交！拿到的任务 ID: {task_ids}")
print("[前端] 用户可以继续做其他事情了...\n")

# 等待所有任务完成
worker_thread.join()

print("\n[前端] 查询所有任务的结果:")
for tid in task_ids:
    print(f"  任务 {tid}: {task_results[tid]}")

**关键观察**：
- 生产者提交任务是**瞬间完成**的（不需要等待处理）
- 消费者在后台**逐个处理**任务
- 前端拿到任务 ID 后就可以去做别的事情了

---

## 9.3 任务链（Pipeline）

在真实的 AI 记忆系统中，处理一条用户数据不是「一步到位」的，而是分成多个步骤串成一条**任务链**（Pipeline）。

### 你们系统的三步任务链

```
ingest_usage          →    persist_fact_memory    →    update_memory_layers
(摄入原始数据)              (提取并保存事实)              (更新记忆层)
```

### 为什么要拆成三步？

你可能会问：为什么不写一个大函数把三件事都做了？拆分的好处：

| 好处 | 说明 |
|------|------|
| **独立失败和重试** | 如果第 2 步 AI 提取失败了，只需要重试第 2 步，前面 OCR 的结果还在 |
| **并行处理** | 当任务 A 在跑第 3 步时，任务 B 可以同时跑第 1 步 |
| **监控粒度** | 可以知道任务卡在哪一步了 |
| **灵活组合** | 不同场景可以只跑部分步骤 |

类比前端：就像 **Promise 链** 或 **RxJS pipe**：
```javascript
// 前端的 Promise 链
fetch(url)
  .then(res => res.json())      // 步骤1：解析响应
  .then(data => transform(data)) // 步骤2：转换数据
  .then(result => render(result)) // 步骤3：渲染页面
  .catch(err => handleError(err)) // 某一步失败了单独处理
```

In [ ]:
import time
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class PipelineStep:
    """任务链中的一个步骤"""
    name: str
    handler: Callable
    max_retries: int = 3


@dataclass
class PipelineResult:
    """任务链的执行结果"""
    success: bool
    data: object = None
    failed_step: str = ""
    error: str = ""
    step_timings: dict = field(default_factory=dict)


class TaskPipeline:
    """简单的任务链实现"""
    
    def __init__(self, name: str):
        self.name = name
        self.steps: list[PipelineStep] = []
    
    def add_step(self, name: str, handler: Callable, max_retries: int = 3):
        self.steps.append(PipelineStep(name, handler, max_retries))
        return self  # 支持链式调用
    
    def run(self, initial_data: object) -> PipelineResult:
        """按顺序执行所有步骤"""
        data = initial_data
        timings: dict[str, float] = {}
        
        for step in self.steps:
            print(f"  [{step.name}] 开始执行...")
            
            # 带重试的执行
            success = False
            for attempt in range(1, step.max_retries + 1):
                try:
                    start = time.time()
                    data = step.handler(data)
                    elapsed = time.time() - start
                    timings[step.name] = elapsed
                    print(f"  [{step.name}] 完成 ({elapsed:.1f}s)")
                    success = True
                    break
                except Exception as e:
                    print(f"  [{step.name}] 第 {attempt} 次尝试失败: {e}")
                    if attempt < step.max_retries:
                        print(f"  [{step.name}] 等待重试...")
                        time.sleep(0.5)
            
            if not success:
                return PipelineResult(
                    success=False,
                    failed_step=step.name,
                    error=f"{step.name} 在 {step.max_retries} 次尝试后仍然失败",
                    step_timings=timings
                )
        
        return PipelineResult(success=True, data=data, step_timings=timings)


# 定义三个步骤
def step_ingest(raw_data: dict) -> dict:
    """步骤1：摄入原始数据（OCR + 解析）"""
    time.sleep(1)  # 模拟 OCR
    return {
        **raw_data,
        "extracted_text": "会议讨论了Q3目标，张三负责用户增长，目标DAU 50万",
        "step": "ingested"
    }

def step_persist_facts(ingested_data: dict) -> dict:
    """步骤2：提取并保存事实"""
    time.sleep(1.5)  # 模拟 AI 提取
    facts = [
        {"subject": "张三", "relation": "负责", "object": "用户增长"},
        {"subject": "Q3目标", "relation": "DAU", "object": "50万"},
    ]
    return {**ingested_data, "facts": facts, "step": "facts_persisted"}

def step_update_layers(facts_data: dict) -> dict:
    """步骤3：更新记忆层"""
    time.sleep(1)  # 模拟记忆层更新
    return {**facts_data, "memory_updated": True, "step": "layers_updated"}


# 构建任务链并执行
pipeline = (
    TaskPipeline("记忆处理流水线")
    .add_step("ingest_usage", step_ingest)
    .add_step("persist_fact_memory", step_persist_facts)
    .add_step("update_memory_layers", step_update_layers)
)

print("开始执行任务链:")
start = time.time()
result = pipeline.run({"source": "screenshot", "file": "meeting.png"})
total = time.time() - start

print(f"\n执行结果: {'成功' if result.success else '失败'}")
print(f"总耗时: {total:.1f}s")
print(f"各步骤耗时: {result.step_timings}")
if result.success:
    print(f"提取到的事实: {result.data['facts']}")

In [ ]:
import random

# 演示：某个步骤失败后自动重试

fail_count = 0

def flaky_persist_facts(data: dict) -> dict:
    """一个不太稳定的 AI 提取步骤 — 模拟偶尔失败"""
    global fail_count
    fail_count += 1
    if fail_count <= 2:  # 前两次失败
        raise ConnectionError("AI 服务暂时不可用")
    time.sleep(0.5)
    return {**data, "facts": [{"key": "retry_demo"}], "step": "facts_persisted"}


pipeline_with_retry = (
    TaskPipeline("带重试的流水线")
    .add_step("ingest_usage", step_ingest)
    .add_step("persist_fact_memory", flaky_persist_facts, max_retries=3)  # 最多重试3次
    .add_step("update_memory_layers", step_update_layers)
)

print("执行带重试的任务链（第2步会失败两次然后成功）:")
fail_count = 0
result = pipeline_with_retry.run({"source": "test"})
print(f"\n最终结果: {'成功' if result.success else '失败'}")
print("重试机制让任务最终成功完成了！")

---
## 9.4 用 Python 模拟你们的记忆任务链

下面用 `asyncio` 来演示异步处理的优势：多个任务可以**并发执行**不同步骤。

### 同步 vs 异步对比

In [ ]:
import asyncio
import time


# ========== 模拟三个处理步骤 ==========

async def step1_ingest(task_id: str) -> dict:
    """步骤1: 摄入数据（模拟 OCR 识别）"""
    print(f"  [任务{task_id}] 步骤1: OCR 识别中...")
    await asyncio.sleep(2)  # 模拟 2 秒的 OCR
    print(f"  [任务{task_id}] 步骤1: OCR 完成")
    return {"task_id": task_id, "text": f"任务{task_id}的识别文本"}


async def step2_persist_facts(data: dict) -> dict:
    """步骤2: 提取并保存事实"""
    task_id = data["task_id"]
    print(f"  [任务{task_id}] 步骤2: AI 提取事实中...")
    await asyncio.sleep(3)  # 模拟 3 秒的 AI 处理
    print(f"  [任务{task_id}] 步骤2: 事实提取完成")
    return {**data, "facts": ["fact1", "fact2"]}


async def step3_update_layers(data: dict) -> dict:
    """步骤3: 更新记忆层"""
    task_id = data["task_id"]
    print(f"  [任务{task_id}] 步骤3: 更新记忆层...")
    await asyncio.sleep(2)  # 模拟 2 秒的更新
    print(f"  [任务{task_id}] 步骤3: 记忆层更新完成")
    return {**data, "memory_updated": True}


async def process_one_task(task_id: str) -> dict:
    """串行执行一个任务的三个步骤"""
    data = await step1_ingest(task_id)
    data = await step2_persist_facts(data)
    data = await step3_update_layers(data)
    return data


# ========== 同步方式：逐个处理 ==========
async def process_sync_style():
    """同步风格：一个做完再做下一个"""
    print("【同步方式】逐个处理 3 个任务")
    print("=" * 45)
    start = time.time()
    
    for i in range(1, 4):
        await process_one_task(str(i))
    
    elapsed = time.time() - start
    print(f"\n同步总耗时: {elapsed:.1f} 秒")
    return elapsed


# ========== 异步方式：并发处理 ==========
async def process_async_style():
    """异步风格：多个任务并发执行"""
    print("\n【异步方式】并发处理 3 个任务")
    print("=" * 45)
    start = time.time()
    
    # 三个任务同时启动！
    tasks = [process_one_task(str(i)) for i in range(1, 4)]
    await asyncio.gather(*tasks)
    
    elapsed = time.time() - start
    print(f"\n异步总耗时: {elapsed:.1f} 秒")
    return elapsed


# 运行对比
sync_time = await process_sync_style()
async_time = await process_async_style()

print("\n" + "=" * 45)
print(f"对比结果:")
print(f"  同步: {sync_time:.1f} 秒 (3个任务 × 7秒 = ~21秒)")
print(f"  异步: {async_time:.1f} 秒 (3个任务并发 = ~7秒)")
print(f"  提速: {sync_time / async_time:.1f}x")

**关键观察**：
- 同步方式：3 个任务 × 7 秒 = 21 秒
- 异步方式：3 个任务并发执行，总共只要约 7 秒
- 因为在一个任务等待 I/O（比如等 AI 返回结果）的时候，另一个任务可以开始处理

---

## 9.5 任务状态与监控

后台任务有一个关键问题：**前端怎么知道任务完成了？**

### 任务状态机

```
pending  →  in_progress  →  completed
                 ↓
              failed  →  (重试) → in_progress
```

在你们的系统里，`Usage` 模型有一个 `memory_status` 字段，就是用来跟踪这个状态的。

### 前端获取状态的三种方式

| 方式 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **轮询 Polling** | 前端每隔几秒请求一次 | 实现最简单 | 浪费请求、有延迟 |
| **WebSocket** | 服务端主动推送 | 实时性好 | 需要维护连接 |
| **SSE** | 服务端单向推送 | 比 WebSocket 简单 | 只能服务端→客户端 |

In [ ]:
import asyncio
import time
from enum import Enum


class TaskStatus(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"


class TaskTracker:
    """任务状态追踪器 — 模拟你们系统的 Usage.memory_status"""
    
    def __init__(self):
        self.tasks: dict[str, dict] = {}
    
    def create_task(self, task_id: str, description: str) -> dict:
        task = {
            "id": task_id,
            "description": description,
            "status": TaskStatus.PENDING.value,
            "current_step": None,
            "progress": 0,
            "created_at": time.time(),
            "result": None,
            "error": None,
        }
        self.tasks[task_id] = task
        return task
    
    def update_status(self, task_id: str, status: TaskStatus, 
                      step: str = None, progress: int = None,
                      result: object = None, error: str = None):
        task = self.tasks[task_id]
        task["status"] = status.value
        if step:
            task["current_step"] = step
        if progress is not None:
            task["progress"] = progress
        if result:
            task["result"] = result
        if error:
            task["error"] = error
    
    def get_status(self, task_id: str) -> dict:
        """查询任务状态 — 这就是轮询时调用的接口"""
        return self.tasks.get(task_id, {"error": "任务不存在"})


# 创建追踪器
tracker = TaskTracker()


async def background_process(task_id: str):
    """后台处理任务，同时更新状态"""
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="ingest", progress=0)
    
    # 步骤 1
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="ingest_usage", progress=10)
    await asyncio.sleep(2)
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="ingest_usage", progress=33)
    
    # 步骤 2
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="persist_fact_memory", progress=40)
    await asyncio.sleep(3)
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="persist_fact_memory", progress=66)
    
    # 步骤 3
    tracker.update_status(task_id, TaskStatus.IN_PROGRESS, step="update_memory_layers", progress=70)
    await asyncio.sleep(2)
    tracker.update_status(
        task_id, TaskStatus.COMPLETED, 
        step="done", progress=100,
        result={"facts_count": 5, "layers_updated": 3}
    )


async def frontend_polling(task_id: str, interval: float = 1.0):
    """前端轮询 — 每隔一段时间查询任务状态"""
    print(f"[前端] 开始轮询任务 {task_id} 的状态（每 {interval} 秒一次）\n")
    
    while True:
        status = tracker.get_status(task_id)
        step = status.get('current_step', '-')
        progress = status.get('progress', 0)
        bar = '█' * (progress // 5) + '░' * (20 - progress // 5)
        print(f"  [轮询] 状态: {status['status']:12s} | 步骤: {step:25s} | 进度: {bar} {progress}%")
        
        if status["status"] in ("completed", "failed"):
            break
        
        await asyncio.sleep(interval)
    
    print(f"\n[前端] 任务完成！结果: {status.get('result')}")


# 模拟完整流程
task_id = "usage_001"
tracker.create_task(task_id, "处理会议截图")
print(f"[API] 创建任务 {task_id}，立即返回给前端")
print(f"[API] 响应: {{\"task_id\": \"{task_id}\", \"status\": \"pending\"}}\n")

# 后台任务和前端轮询同时运行
await asyncio.gather(
    background_process(task_id),
    frontend_polling(task_id, interval=1.0)
)

**关键要点**：
- 后台任务在处理的同时不断更新状态
- 前端通过轮询定期查看进度
- 进度条让用户知道「系统在干活」，而不是以为卡住了

在你们的系统中，`Usage.memory_status` 字段就起到了这个作用：
- `pending`: 刚创建，还没开始处理
- `in_progress`: 正在处理中（对应上面的各个步骤）
- `completed`: 处理完成
- `failed`: 处理失败

---

## 9.6 你们系统的具体实现

### Dramatiq 框架简介

你们的系统使用 **Dramatiq** 来处理后台任务。Dramatiq 是一个 Python 任务队列框架，类似于更知名的 Celery，但更简洁。

```python
# 这是你们系统中的大致写法（简化版）
import dramatiq

@dramatiq.actor(max_retries=0)  # <-- 注意这里！
def ingest_usage(usage_id: int):
    usage = Usage.objects.get(id=usage_id)
    # ... 执行摄入逻辑
    persist_fact_memory.send(usage_id)  # 触发下一步

@dramatiq.actor(max_retries=0)
def persist_fact_memory(usage_id: int):
    # ... 提取并保存事实
    update_memory_layers.send(usage_id)  # 触发下一步

@dramatiq.actor(max_retries=0)
def update_memory_layers(usage_id: int):
    # ... 更新记忆层
    pass
```

### 对照前端概念理解

| Dramatiq 概念 | 前端类比 |
|---------------|----------|
| `@dramatiq.actor` | 注册一个异步任务（类似注册 Service Worker 的 handler） |
| `.send()` | 发送任务到队列（类似 `postMessage()` 或 `fetch()`） |
| Redis Broker | 消息中转站（类似 Redux Store 或 Event Bus） |
| Worker 进程 | 实际执行任务的进程（类似 Web Worker） |

### 问题：max_retries=0

注意看上面代码中的 `max_retries=0`，这意味着**任务失败后不会重试**。这在生产环境中是一个隐患。

In [ ]:
import asyncio
import random


# 模拟 Dramatiq 的 actor 装饰器行为
class SimpleTaskQueue:
    """简化版任务队列 — 模拟 Dramatiq 的核心行为"""
    
    def __init__(self):
        self.actors: dict[str, dict] = {}  # 注册的 actor
        self.results: list[str] = []  # 执行日志
    
    def actor(self, max_retries: int = 0):
        """装饰器：注册一个 actor（类似 @dramatiq.actor）"""
        def decorator(func):
            self.actors[func.__name__] = {
                "handler": func,
                "max_retries": max_retries,
            }
            
            # 给函数添加 .send() 方法
            async def send(*args, **kwargs):
                actor_info = self.actors[func.__name__]
                handler = actor_info["handler"]
                retries = actor_info["max_retries"]
                
                for attempt in range(retries + 1):
                    try:
                        result = await handler(*args, **kwargs)
                        self.results.append(f"{func.__name__}: 成功")
                        return result
                    except Exception as e:
                        self.results.append(f"{func.__name__}: 第{attempt+1}次失败 - {e}")
                        if attempt < retries:
                            print(f"    ↻ 重试 {func.__name__} ({attempt+2}/{retries+1})")
                            await asyncio.sleep(0.5)
                        else:
                            print(f"    ✗ {func.__name__} 最终失败！")
                            raise
            
            func.send = send
            return func
        return decorator


# ========== 场景1: max_retries=0（你们当前的配置） ==========
print("场景1: max_retries=0（当前配置 — 不重试）")
print("=" * 50)

queue_no_retry = SimpleTaskQueue()

@queue_no_retry.actor(max_retries=0)
async def ingest_v1(usage_id: int):
    print(f"  ingest_usage({usage_id}): 处理中...")
    await asyncio.sleep(0.5)
    return True

@queue_no_retry.actor(max_retries=0)
async def persist_v1(usage_id: int):
    print(f"  persist_fact_memory({usage_id}): 调用 AI 服务...")
    # 模拟 AI 服务偶尔超时
    if random.random() < 0.7:  # 70% 概率失败
        raise TimeoutError("AI 服务响应超时")
    return True

# 模拟运行 5 次
random.seed(42)  # 固定随机数以便重现
success_count = 0
for i in range(5):
    try:
        await ingest_v1.send(i)
        await persist_v1.send(i)
        success_count += 1
        print(f"  → 任务 {i} 成功!\n")
    except TimeoutError:
        print(f"  → 任务 {i} 失败，数据丢失！\n")

print(f"结果: {success_count}/5 成功。失败的任务数据永远丢失了！")

In [ ]:
# ========== 场景2: max_retries=3（改进后的配置） ==========
print("场景2: max_retries=3（改进后 — 自动重试）")
print("=" * 50)

queue_with_retry = SimpleTaskQueue()

@queue_with_retry.actor(max_retries=3)
async def ingest_v2(usage_id: int):
    print(f"  ingest_usage({usage_id}): 处理中...")
    await asyncio.sleep(0.3)
    return True

@queue_with_retry.actor(max_retries=3)
async def persist_v2(usage_id: int):
    print(f"  persist_fact_memory({usage_id}): 调用 AI 服务...")
    # 同样 70% 概率失败，但有重试机会
    if random.random() < 0.7:
        raise TimeoutError("AI 服务响应超时")
    return True

random.seed(42)  # 同样的随机种子
success_count = 0
for i in range(5):
    try:
        await ingest_v2.send(i)
        await persist_v2.send(i)
        success_count += 1
        print(f"  → 任务 {i} 成功!\n")
    except TimeoutError:
        print(f"  → 任务 {i} 最终失败（重试3次仍然不行）\n")

print(f"结果: {success_count}/5 成功。重试机制大大提高了成功率！")
print("\n建议: 将 max_retries 设为 3，并加上指数退避 (exponential backoff)")

In [ ]:
import asyncio

# ========== 指数退避策略演示 ==========
print("指数退避（Exponential Backoff）策略")
print("=" * 50)
print("\n每次重试的等待时间翻倍，避免雪崩式重试：\n")


async def retry_with_backoff(func, max_retries: int = 3, base_delay: float = 1.0):
    """带指数退避的重试策略"""
    for attempt in range(max_retries + 1):
        try:
            return await func()
        except Exception as e:
            if attempt == max_retries:
                print(f"  第 {attempt + 1} 次失败: {e} — 放弃重试")
                raise
            
            delay = base_delay * (2 ** attempt)  # 1s, 2s, 4s...
            print(f"  第 {attempt + 1} 次失败: {e} — 等待 {delay}s 后重试")
            await asyncio.sleep(delay * 0.1)  # 缩短时间以便演示


# 模拟不稳定的服务
call_count = 0

async def unstable_ai_service():
    global call_count
    call_count += 1
    if call_count < 4:  # 前3次失败，第4次成功
        raise ConnectionError("AI 服务繁忙")
    return "提取到 3 条事实"


call_count = 0
try:
    result = await retry_with_backoff(unstable_ai_service, max_retries=4)
    print(f"\n最终成功! 结果: {result}")
except Exception:
    print("\n最终失败!")

print(f"\n退避时间表:")
for i in range(5):
    print(f"  第 {i+1} 次重试: 等待 {1 * (2**i)} 秒")
print("  → 这样可以给服务恢复的时间，避免反复冲击已经过载的服务")

### 改进建议总结

| 当前实现 | 问题 | 建议改进 |
|---------|------|----------|
| `max_retries=0` | 任何瞬时故障都会导致任务永久失败 | 改为 `max_retries=3` |
| 无退避策略 | 立即重试可能加重服务负担 | 加上指数退避 |
| 失败无告警 | 运维不知道任务失败了 | 加监控和告警 |

```python
# 改进后的写法
@dramatiq.actor(
    max_retries=3,
    min_backoff=1000,    # 最小退避 1 秒
    max_backoff=60000,   # 最大退避 60 秒
)
def persist_fact_memory(usage_id: int):
    # ...
```

---

## 9.7 本课小结

### 核心概念回顾

| 概念 | 一句话解释 | 前端类比 |
|------|-----------|----------|
| **异步任务** | 前端下单后不等结果，后台慢慢处理 | `fetch()` 返回 Promise，不阻塞 UI |
| **消息队列** | 任务的中转站，解耦生产者和消费者 | Event Bus / Redux Store |
| **任务链** | 多个步骤串成流水线，每步可独立失败重试 | Promise 链 / RxJS pipe |
| **状态跟踪** | 让前端知道后台任务的进度 | loading 状态管理 |
| **重试策略** | 任务失败后自动重试，带退避机制 | fetch 重试 / axios retry |

### 你们系统的任务流

```
用户上传截图
    ↓
API 返回 task_id（用户不用等）
    ↓
后台任务链:
    ingest_usage → persist_fact_memory → update_memory_layers
    ↓
前端轮询 Usage.memory_status 获取进度
    ↓
status = completed → 前端展示结果
```

### 下一课预告

下一课我们将学习**记忆的检索与排序** — 当记忆越来越多，怎么快速找到最相关的那条？

In [ ]:
# 综合练习：把本课学到的内容串起来
# 实现一个完整的「后台任务系统」迷你版

import asyncio
import time
import uuid
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Awaitable


class Status(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"


@dataclass
class Task:
    id: str
    status: Status = Status.PENDING
    current_step: str = ""
    progress: int = 0
    result: object = None
    error: str = ""


class MiniTaskSystem:
    """迷你后台任务系统 — 整合队列 + 任务链 + 状态追踪"""
    
    def __init__(self):
        self.tasks: dict[str, Task] = {}
        self.queue: asyncio.Queue = asyncio.Queue()
        self.steps: list[tuple[str, Callable[..., Awaitable], int]] = []  # (name, handler, retries)
    
    def add_step(self, name: str, handler: Callable[..., Awaitable], max_retries: int = 3):
        self.steps.append((name, handler, max_retries))
        return self
    
    async def submit(self, data: dict) -> str:
        """提交任务 — 立刻返回任务 ID"""
        task_id = str(uuid.uuid4())[:8]
        self.tasks[task_id] = Task(id=task_id)
        await self.queue.put((task_id, data))
        return task_id
    
    def get_status(self, task_id: str) -> dict:
        task = self.tasks.get(task_id)
        if not task:
            return {"error": "not found"}
        return {
            "id": task.id,
            "status": task.status.value,
            "step": task.current_step,
            "progress": task.progress,
        }
    
    async def _process_task(self, task_id: str, data: dict):
        """按步骤执行任务链"""
        task = self.tasks[task_id]
        task.status = Status.IN_PROGRESS
        
        for i, (name, handler, max_retries) in enumerate(self.steps):
            task.current_step = name
            task.progress = int((i / len(self.steps)) * 100)
            
            success = False
            for attempt in range(max_retries + 1):
                try:
                    data = await handler(data)
                    success = True
                    break
                except Exception as e:
                    if attempt < max_retries:
                        delay = 0.1 * (2 ** attempt)  # 指数退避
                        await asyncio.sleep(delay)
                    else:
                        task.status = Status.FAILED
                        task.error = f"{name} 失败: {e}"
                        return
        
        task.status = Status.COMPLETED
        task.progress = 100
        task.current_step = "done"
        task.result = data
    
    async def run_worker(self, worker_id: int = 1):
        """Worker：从队列取任务并处理"""
        while True:
            try:
                task_id, data = await asyncio.wait_for(self.queue.get(), timeout=3)
                print(f"[Worker-{worker_id}] 处理任务 {task_id}")
                await self._process_task(task_id, data)
                self.queue.task_done()
            except asyncio.TimeoutError:
                break


# ========== 使用迷你任务系统 ==========

system = MiniTaskSystem()

# 定义处理步骤
async def ingest(data: dict) -> dict:
    await asyncio.sleep(0.5)
    return {**data, "text": "OCR结果"}

async def persist(data: dict) -> dict:
    await asyncio.sleep(0.8)
    return {**data, "facts": ["张三负责增长"]}

async def update(data: dict) -> dict:
    await asyncio.sleep(0.5)
    return {**data, "updated": True}

system.add_step("ingest_usage", ingest, max_retries=3)
system.add_step("persist_fact_memory", persist, max_retries=3)
system.add_step("update_memory_layers", update, max_retries=3)

# 提交多个任务
task_ids = []
for i in range(3):
    tid = await system.submit({"file": f"screenshot_{i+1}.png"})
    task_ids.append(tid)
    print(f"提交任务: {tid}")

print(f"\n所有任务已提交，任务 ID: {task_ids}")
print("启动 2 个 Worker 并发处理...\n")

# 启动 2 个 Worker 并发处理
start = time.time()
await asyncio.gather(
    system.run_worker(1),
    system.run_worker(2),
)
elapsed = time.time() - start

print(f"\n全部完成！耗时 {elapsed:.1f} 秒")
print("\n各任务最终状态:")
for tid in task_ids:
    status = system.get_status(tid)
    print(f"  任务 {tid}: {status['status']} (进度 {status['progress']}%)")

**恭喜完成第 9 课！** 现在你理解了：

1. 为什么需要后台任务（用户不能等 20 秒）
2. 消息队列的生产者-消费者模型（餐厅点餐）
3. 任务链的设计（拆分步骤、独立重试）
4. 状态追踪和前端轮询
5. 重试策略和指数退避
6. 你们系统中 Dramatiq 的实际用法和改进方向